# Colab 전체 학습·검증·테스트

이 노트북을 위에서부터 실행하면 다음 순서로 진행합니다.

1. Google Drive 연결 및 프로젝트 자동 탐색
2. 학습 환경 설치와 GPU 확인
3. 2022년 안양대 격자 정답 확인·수집
4. 학습 데이터 생성
5. 5개 모델 학습(2022~2023)·검증(2024)
6. 2024년 검증 성능 비교표 확인
7. 검증 종합 점수로 best 모델 선택
8. 선택을 고정한 채 모든 모델의 2025년 및 가능한 경우 2026년 평가
9. 검증 1위 모델을 로컬호스트와 정적 사이트에 배포·검증

> 실행 전 Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요.

## 1. Drive 연결 및 프로젝트 찾기

`PROJECT_DIR`를 비워 두면 Drive 안에서 저장소를 자동으로 찾습니다. 같은 이름의 저장소가 여러 개라면 정확한 Colab 경로를 입력하세요.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

PROJECT_DIR = ''  # 예: /content/drive/MyDrive/annyang/perceived-temperature-forAyu
if PROJECT_DIR:
    project_dir = Path(PROJECT_DIR).expanduser().resolve()
    candidates = [project_dir]
else:
    candidates = [
        p.parent for p in Path('/content/drive/MyDrive').rglob('requirements-train.txt')
        if p.parent.name == 'perceived-temperature-forAyu'
    ]

valid = [p for p in candidates if (p / 'train_model.py').is_file()]
if len(valid) != 1:
    raise RuntimeError(f'프로젝트를 하나만 찾을 수 있어야 합니다. PROJECT_DIR을 직접 지정하세요: {valid}')

PROJECT_DIR = valid[0]
os.chdir(PROJECT_DIR)
print('프로젝트 폴더:', PROJECT_DIR)
print('현재 폴더:', Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
프로젝트 폴더: /content/drive/MyDrive/annyang/perceived-temperature-forAyu
현재 폴더: /content/drive/MyDrive/annyang/perceived-temperature-forAyu


## 2. 환경 설치 및 GPU 확인

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-train.txt'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA 사용 가능:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음(CPU 사용)')

PyTorch: 2.14.1+cu130
CUDA 사용 가능: True
GPU: Tesla T4


## 3. 2022년 안양대 격자 정답 준비

이미 `data/grid/grid_2022_01.csv`~`grid_2022_12.csv`가 있으면 수집을 건너뜁니다. 없는 달이 있으면 Colab 왼쪽 **열쇠 아이콘(Secrets)**에 이름 `KMA_API_KEY`로 저장한 값을 사용합니다. Secret이 없을 때만 비공개 입력창을 표시하며 인증키는 노트북 파일에 저장하지 않습니다.

In [ ]:
import time
from datetime import datetime, timedelta
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests

GRID_DIR = PROJECT_DIR / 'data' / 'grid'
GRID_DIR.mkdir(parents=True, exist_ok=True)
expected = [GRID_DIR / f'grid_2022_{month:02d}.csv' for month in range(1, 13)]
missing = [path for path in expected if not path.is_file()]
print(f'2022 격자: {12 - len(missing)}/12개월 준비됨')

if missing:
    try:
        from google.colab import userdata
        AUTH_KEY = (userdata.get('KMA_API_KEY') or '').strip()
        print('Colab Secret KMA_API_KEY를 사용합니다.')
    except Exception:
        AUTH_KEY = ''
    if not AUTH_KEY:
        AUTH_KEY = getpass('기상청 API허브 인증키(화면에 표시되지 않음): ').strip()
    if not AUTH_KEY:
        raise RuntimeError('2022년 격자 수집에 API허브 인증키가 필요합니다.')

    URL = 'https://apihub.kma.go.kr/api/typ01/url/sfc_nc_var.php'
    LAT, LON = 37.3921, 126.9206
    OBS = 'ta,hm,ws_10m,ta_chi'
    COLS = ['tm'] + OBS.split(',')
    WORKERS = 4

    def fetch(tm1, tm2):
        params = {
            'tm1': tm1, 'tm2': tm2, 'obs': OBS, 'itv': '60',
            'lon': LON, 'lat': LAT, 'help': '0', 'authKey': AUTH_KEY,
        }
        response = requests.get(URL, params=params, timeout=(10, 30))
        response.raise_for_status()
        rows = []
        for line in response.text.splitlines():
            parts = line.strip().replace(',', ' ').split()
            if len(parts) >= len(COLS) and parts[0].isdigit():
                rows.append(parts[:len(COLS)])
        frame = pd.DataFrame(rows, columns=COLS)
        if frame.empty:
            return frame
        frame['tm'] = pd.to_datetime(frame['tm'], format='%Y%m%d%H%M')
        for column in COLS[1:]:
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
            frame.loc[frame[column] <= -50, column] = np.nan
        return frame.dropna(subset=['ta'])

    def fetch_window(start):
        tm1 = start.strftime('%Y%m%d%H%M')
        tm2 = (start + timedelta(hours=1)).strftime('%Y%m%d%H%M')
        for attempt in range(4):
            try:
                result = fetch(tm1, tm2)
                if not result.empty:
                    return result
            except Exception:
                pass
            time.sleep(2 * (attempt + 1))
        return None

    def fetch_month(year, month):
        start = datetime(year, month, 1)
        stop = datetime(year + (month == 12), month % 12 + 1, 1)
        windows = []
        cursor = start
        while cursor < stop:
            windows.append(cursor)
            cursor += timedelta(hours=2)
        results = []
        with ThreadPoolExecutor(max_workers=WORKERS) as executor:
            futures = [executor.submit(fetch_window, window) for window in windows]
            for index, future in enumerate(as_completed(futures), start=1):
                results.append(future.result())
                if index % max(1, len(futures) // 4) == 0:
                    print(f'  {year}-{month:02d}: {index * 100 // len(futures)}%')
        failed = sum(result is None for result in results)
        if failed > len(windows) * 0.1:
            raise RuntimeError(f'{year}-{month:02d}: {len(windows)}개 구간 중 {failed}개 실패')
        frame = pd.concat([result for result in results if result is not None], ignore_index=True)
        frame = frame[(frame.tm >= start) & (frame.tm < stop)]
        return frame.drop_duplicates('tm').sort_values('tm'), failed

    for path in missing:
        month = int(path.stem.rsplit('_', 1)[1])
        started = time.time()
        frame, failed = fetch_month(2022, month)
        frame.to_csv(path, index=False, encoding='utf-8-sig')
        print(f'{path.name}: {len(frame):,}줄 저장, 실패 구간 {failed}, {time.time() - started:.0f}초')

missing_after = [path for path in expected if not path.is_file()]
if missing_after:
    raise RuntimeError(f'수집되지 않은 파일: {missing_after}')
print('2022년 격자 12개월 준비 완료:', GRID_DIR)

## 4. 학습 데이터 생성 및 연도 확인

In [ ]:
subprocess.run([sys.executable, 'prepare_training_data.py', '--grid-dir', 'data/grid'], check=True)

MODEL_DATA = PROJECT_DIR / 'data' / 'prepared' / 'model_data.csv'
if not MODEL_DATA.is_file():
    raise FileNotFoundError(f'학습 데이터 생성 실패: {MODEL_DATA}')

prepared = pd.read_csv(MODEL_DATA, encoding='utf-8-sig', usecols=['tm'])
prepared['year'] = pd.to_datetime(prepared['tm']).dt.year
year_counts = prepared['year'].value_counts().sort_index()
print(year_counts.to_string())
required = {2022, 2023, 2024, 2025}
missing_years = required - set(year_counts.index)
if missing_years:
    raise RuntimeError(f'필수 연도 누락: {sorted(missing_years)}')

## 5. 모델별 학습·2024년 검증

필요하면 `MODELS`에서 일부 모델만 남겨 개별 실행할 수 있습니다. LSTM과 TSMixer는 CUDA가 있으면 GPU를 사용합니다.

In [ ]:
MODELS = ['rf', 'rf-lag', 'lightgbm', 'lstm', 'tsmixer']
SEEDS = 3
DEVICE = 'auto'

for model in MODELS:
    command = [
        sys.executable, 'train_model.py', '--model', model,
        '--data', str(MODEL_DATA), '--device', DEVICE,
    ]
    if model in {'lstm', 'tsmixer'}:
        command += ['--seeds', str(SEEDS)]
    print('\n' + '=' * 70)
    print('학습 시작:', model)
    subprocess.run(command, check=True)
print('모든 모델 학습·검증 완료')

## 6. 2024년 검증 성능 비교표

모든 모델이 같은 2022~2023년 학습 데이터와 2024년 검증 데이터로 평가됐는지 확인하고, 종합점수 내림차순 비교표를 표시합니다. 이 셀에서는 아직 모델을 선택하거나 테스트 데이터를 읽지 않습니다.

In [ ]:
import json
from compare_models import TRAIN_MODEL_NAMES

checkpoint_root = PROJECT_DIR / 'compare_runs' / 'checkpoints'
comparison_rows = []
signatures = set()
for model in MODELS:
    metadata_path = checkpoint_root / model / 'metadata.json'
    if not metadata_path.is_file():
        raise FileNotFoundError(f'{model} 학습 결과가 없습니다: {metadata_path}')
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    if metadata.get('status') != 'completed':
        raise RuntimeError(f'{model} 학습이 완료되지 않았습니다: {metadata.get("status")}')
    settings = metadata['settings']
    signatures.add((settings['train_val_sha256'], settings['train_start_year'], settings['val_year']))
    comparison_rows.append({
        'model_key': model,
        '모델': TRAIN_MODEL_NAMES[model],
        **metadata['validation_metrics'],
        '학습 시간(초)': metadata['training_seconds'],
    })

if len(signatures) != 1:
    raise RuntimeError('모델마다 학습·검증 데이터 또는 연도 설정이 다릅니다.')

validation_table = (
    pd.DataFrame(comparison_rows)
    .sort_values('종합 점수', ascending=False)
    .reset_index(drop=True)
)
validation_table.index = validation_table.index + 1
validation_table.index.name = '순위'
metric_formats = {
    '종합 점수': '{:.2f}', '기온 오차(°C)': '{:.3f}', '습도 오차(%)': '{:.3f}',
    '풍속 오차(m/s)': '{:.3f}', '체감 오차(°C)': '{:.3f}',
    '±1°C 적중(%)': '{:.2f}', '±2°C 적중(%)': '{:.2f}',
    '겨울 ±1°C(%)': '{:.2f}', '학습 시간(초)': '{:.1f}',
}
# 다크모드에서도 보이도록 최고 점수 색상을 직접 지정합니다.
# 원하는 색으로 바꾸려면 아래 두 HEX 값만 수정하세요.
BEST_SCORE_BG = '#FFD54F'    # 최고 점수 배경: 노란색
BEST_SCORE_TEXT = '#111827'  # 최고 점수 글자: 진한 남색
BEST_SCORE_STYLE = f'background-color: {BEST_SCORE_BG}; color: {BEST_SCORE_TEXT}; font-weight: 700;'

overall_score_table = validation_table[['모델', '종합 점수']].copy()
print('2024년 검증 종합점수 순위')
display(overall_score_table.style.format({'종합 점수': '{:.2f}'}).highlight_max(subset=['종합 점수'], props=BEST_SCORE_STYLE))
print('2024년 검증 상세 지표')
display(validation_table.style.format(metric_formats).highlight_max(subset=['종합 점수'], props=BEST_SCORE_STYLE))

2024년 검증 종합점수 순위


,모델,종합 점수
순위,,
1,LightGBM + 지연특성,83.89
2,TSMixer,82.64
3,RF + 지연특성,82.46
4,LSTM,81.92
5,RF (지금 운영),70.93


2024년 검증 상세 지표


,model_key,모델,종합 점수,기온 오차(°C),습도 오차(%),풍속 오차(m/s),체감 오차(°C),±1°C 적중(%),사계절 평균 ±1°C(%),봄(3~5월) ±1°C(%),여름(6~8월) ±1°C(%),가을(9~11월) ±1°C(%),겨울(12~2월) ±1°C(%),±2°C 적중(%),겨울공식 기간(10~4월) ±1°C(%),학습 시간(초)
순위,,,,,,,,,,,,,,,,
1,lightgbm,LightGBM + 지연특성,83.89,0.576,4.257,0.346,0.608,81.95,81.927511,85.086129,90.727273,82.052462,69.844180,97.16,76.005493,16.8
2,tsmixer,TSMixer,82.64,0.594,3.730,0.328,0.635,80.46,80.429391,83.091568,91.409091,80.809940,66.406966,96.66,73.082205,24.7
3,rf-lag,RF + 지연특성,82.46,0.619,3.900,0.350,0.642,80.32,80.293441,83.227561,90.090909,80.027612,67.827681,96.38,73.376496,78.6
4,lstm,LSTM,81.92,0.613,3.759,0.331,0.642,79.59,79.560868,81.731641,92.045455,80.625863,63.840513,96.14,71.257603,22.2
5,rf,RF (지금 운영),70.93,0.897,4.820,0.374,0.899,67.06,67.026820,67.860381,80.590909,65.485504,54.170486,90.00,56.955072,8.5


## 7. 2024년 검증 1위 모델 확정

위 표의 종합점수 1위를 `selection.json`에 고정합니다. 이 셀도 2025·2026년 테스트 데이터는 읽지 않습니다.

In [ ]:
subprocess.run([sys.executable, 'compare_checkpoints.py', '--select-only', '--models', *MODELS], check=True)

selection_path = checkpoint_root / 'selection.json'
selection = json.loads(selection_path.read_text(encoding='utf-8'))
selected_key = selection['selected_model']
expected_key = validation_table.iloc[0]['model_key']
if selected_key != expected_key:
    raise RuntimeError(f'표의 1위({expected_key})와 선택 결과({selected_key})가 다릅니다.')
print('최종 선택 모델:', selection['selected_model_name'])
print('2024년 검증 종합점수:', selection['validation_overall_score'])
print('체크포인트:', selection['checkpoint'])

최종 선택 모델: LightGBM + 지연특성
2024년 검증 종합점수: 83.89198579995228
체크포인트: lightgbm/best.joblib


## 8. 선택을 고정한 뒤 모든 모델 테스트

2024년 검증 1위는 그대로 고정하고, `MODELS`의 모든 모델을 2025년에 평가합니다. 2026년 정답 행이 준비된 경우에도 같은 방식으로 별도 평가합니다. 테스트 결과를 보고 모델을 다시 선택하지 않습니다.

In [ ]:
available_years = set(year_counts.index.astype(int))
for test_year in [2025, 2026]:
    if test_year not in available_years:
        print(f'{test_year}년 정답 데이터가 없어 평가를 건너뜁니다.')
        continue
    subprocess.run([
        sys.executable, 'compare_checkpoints.py',
        '--data', str(MODEL_DATA), '--test-year', str(test_year),
        '--device', DEVICE, '--models', *MODELS,
    ], check=True)

for year in [2025, 2026]:
    result = PROJECT_DIR / 'compare_runs' / 'checkpoints' / f'comparison_{year}.csv'
    if result.is_file():
        print(f'\n{year}년 결과')
        test_table = pd.read_csv(result, encoding='utf-8-sig').sort_values('종합 점수', ascending=False)
        display(test_table.style.format(metric_formats).highlight_max(subset=['종합 점수'], props=BEST_SCORE_STYLE))

2026년 정답 데이터가 없어 평가를 건너뜁니다.

2025년 결과


,model,종합 점수,±1°C 적중(%),사계절 평균 ±1°C(%),±2°C 적중(%),겨울공식 기간(10~4월) ±1°C(%),체감 오차(°C),기온 오차(°C),습도 오차(%),풍속 오차(m/s),RF 대비(%p),학습 시간(초),학습 정보,검증 종합 점수,2024 검증 선택,체크포인트
4,LSTM,84.94,83.39,83.341832,96.57,76.556922,0.580,0.537,3.829,0.321,12.109734,22.2,검증 81.9 · best.pt,81.917609,False,best.pt
3,LightGBM + 지연특성,84.86,83.29,83.242841,96.49,76.239587,0.581,0.521,4.192,0.339,12.006428,16.8,2024 선택 · 검증 83.9 · best.joblib,83.891986,True,best.joblib
5,TSMixer,84.71,83.14,83.090916,96.66,76.418088,0.594,0.529,3.861,0.327,11.857208,24.7,검증 82.6 · best.pt,82.638008,False,best.pt
2,RF + 지연특성,83.32,81.55,81.521231,95.45,75.823086,0.616,0.557,4.213,0.344,10.273186,78.6,검증 82.5 · best.joblib,82.462222,False,best.joblib
1,RF (지금 운영),74.35,71.28,71.216157,90.40,63.149544,0.818,0.789,5.066,0.368,0.000000,8.5,검증 70.9 · best.joblib,70.934619,False,best.joblib
0,보정 안 함(날씨앱),57.56,56.80,56.689465,71.89,46.747322,1.590,1.658,8.450,0.521,-14.485767,nan,-,nan,False,-


## 9. 최고 모델 앱·정적 사이트 연동 확인

2024년 검증 1위 체크포인트를 Flask 로컬호스트용으로 배포하고, 같은 모델을 `docs/model.js`로 내보냅니다. 마지막으로 Python 체크포인트와 정적 브라우저 모델의 예측값이 일치하는지 검사합니다.

In [ ]:
subprocess.run([sys.executable, 'deploy_best_model.py', '--data', str(MODEL_DATA)], check=True)
subprocess.run([sys.executable, 'web_export.py'], check=True)
subprocess.run([
    sys.executable, 'verify_static_export.py',
    '--manifest', 'data/deployed_campus_model/manifest.json',
    '--model-js', 'docs/model.js', '--data', str(MODEL_DATA),
], check=True)

from server import app
status_response = app.test_client().get('/api/model-status')
model_status = status_response.get_json()
display(model_status)
assert status_response.status_code == 200 and model_status['deployed']
assert model_status['selected_model'] == selection['selected_model']
print('로컬호스트 연동 확인 완료:', model_status['selected_model_name'])
print('정적 사이트 모델 생성 완료: docs/model.js')

{'deployed': True,
 'has_rf_fallback': True,
 'history_hours': 24,
 'mode': 'selected_checkpoint',
 'score_version': 'overall_v2_season_balanced',
 'selected_model': 'lightgbm',
 'selected_model_name': 'LightGBM + 지연특성',
 'validation_overall_score': 83.89198579995228,
 'validation_year': 2024}

로컬호스트 연동 확인 완료: LightGBM + 지연특성
정적 사이트 모델 생성 완료: docs/model.js


## 10. Flask 로컬호스트 실행

최고 모델이 연결된 실제 Flask 사이트를 백그라운드로 실행합니다. Colab에서는 `localhost`가 사용자 PC가 아니라 원격 런타임을 뜻하므로 아래 셀이 Colab 전용 프록시를 사용합니다. `APP_VIEW = 'window'`는 새 탭 링크를 만들고, `'iframe'`은 노트북 안에 화면을 표시합니다. 이 화면과 링크는 현재 노트북 런타임이 연결된 동안만 사용할 수 있습니다.

Colab의 새 창 기능은 브라우저 보안 설정에 따라 열리지 않을 수 있습니다. 그때는 `APP_VIEW = 'iframe'`으로 바꾸고 셀을 다시 실행하세요. 사용자 PC의 진짜 로컬호스트로 실행하려면 학습 결과가 동기화된 프로젝트 폴더에서 `python server.py`를 실행하고 `http://127.0.0.1:5000`을 여세요.

In [ ]:
import time
import urllib.request

PORT = 5000
APP_VIEW = 'window'  # 'window': 새 탭 링크, 'iframe': 노트북 안에 표시
APP_LOG_PATH = PROJECT_DIR / 'compare_runs' / 'logs' / 'app' / 'flask_server.log'
APP_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

# 이 셀을 다시 실행할 때 이전 서버와 로그 파일을 안전하게 정리합니다.
if 'FLASK_PROCESS' in globals() and FLASK_PROCESS.poll() is None:
    FLASK_PROCESS.terminate()
    try:
        FLASK_PROCESS.wait(timeout=5)
    except subprocess.TimeoutExpired:
        FLASK_PROCESS.kill()
if 'FLASK_LOG' in globals() and not FLASK_LOG.closed:
    FLASK_LOG.close()

FLASK_LOG = APP_LOG_PATH.open('a', encoding='utf-8')
server_env = os.environ.copy()
server_env['PYTHONUNBUFFERED'] = '1'
FLASK_PROCESS = subprocess.Popen(
    [sys.executable, 'server.py'],
    cwd=PROJECT_DIR, env=server_env,
    stdout=FLASK_LOG, stderr=subprocess.STDOUT,
)

status_url = f'http://127.0.0.1:{PORT}/api/model-status'
for _ in range(60):
    if FLASK_PROCESS.poll() is not None:
        FLASK_LOG.flush()
        raise RuntimeError(f'Flask 서버가 종료됐습니다. 로그를 확인하세요: {APP_LOG_PATH}')
    try:
        with urllib.request.urlopen(status_url, timeout=2) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(0.5)
else:
    FLASK_PROCESS.terminate()
    raise TimeoutError(f'30초 안에 Flask 서버가 시작되지 않았습니다: {APP_LOG_PATH}')

print('Flask 서버 실행 완료:', status_url)
print('서버 로그:', APP_LOG_PATH)
print('종료하려면 새 셀에서 FLASK_PROCESS.terminate() 를 실행하세요.')

try:
    from google.colab import output
    if APP_VIEW == 'window':
        output.serve_kernel_port_as_window(
            PORT, path='/', anchor_text='오늘춥나?? 새 창으로 열기', skip_warning=True,
        )
        print('위 링크가 열리지 않으면 APP_VIEW를 iframe으로 바꾸고 다시 실행하세요.')
    elif APP_VIEW == 'iframe':
        output.serve_kernel_port_as_iframe(PORT, path='/', height=850)
    else:
        raise ValueError("APP_VIEW는 'window' 또는 'iframe'이어야 합니다.")
except ImportError:
    print(f'로컬 브라우저에서 여세요: http://127.0.0.1:{PORT}')

Flask 서버 실행 완료: http://127.0.0.1:5000/api/model-status
서버 로그: /content/drive/MyDrive/annyang/perceived-temperature-forAyu/compare_runs/logs/app/flask_server.log
종료하려면 새 셀에서 FLASK_PROCESS.terminate() 를 실행하세요.


<IPython.core.display.Javascript object>

위 링크가 열리지 않으면 APP_VIEW를 iframe으로 바꾸고 다시 실행하세요.


In [ ]:
FLASK_PROCESS.terminate()